<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/ewpd4lhc_wilson_ray_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EWPD4LHC → Wilson-Ray Inputs (Flavor-Universal Default) + χ²⊥ Test

This Colab notebook:

1. Clones the `ewpd4lhc/ewpd4lhc` repository.
2. Runs the **default** (flavor-universal) configuration to generate `ewpd_out.yml`.
3. Extracts:
   - `coeff_names` (operator ordering),
   - the observable-response matrix `A` (Jacobian),
   - the observable covariance (or its inverse),
   - constructs the coefficient-space Fisher matrix `F = Aᵀ V⁻¹ A`,
   - constructs `SigmaC = pinv(F)` (supported-subspace covariance).
4. Computes the Wilson-ray orthogonal deviation statistic **χ²⊥** using `F` as the quadratic form:
\[
\chi^2_{\perp} = \hat C^{T} F \hat C - \frac{(v^{T} F \hat C)^2}{v^{T} F v}.
\]
5. Saves outputs as `.npy` for download.

This is a one-time extraction + linear-algebra evaluation (no software paper, no framework building).

## 0) Environment check (Colab)

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

## 1) Clone repository

In [ ]:
!git clone https://github.com/ewpd4lhc/ewpd4lhc.git
%cd ewpd4lhc
!ls -la

## 2) Install dependencies (minimal)

In [ ]:
!pip -q install numpy pyyaml scipy pandas

## 3) Run default EWPD build (flavor-universal)

Runs the repository script without arguments. Expected output file: `ewpd_out.yml`.
If your repo version differs, the file listing will reveal the produced filename.

In [ ]:
!chmod +x ewpd4lhc.py
!./ewpd4lhc.py
!ls -lh

## 4) Load YAML output and inspect keys

In [ ]:
import yaml
from pathlib import Path

yml_path = Path("ewpd_out.yml")
if not yml_path.exists():
    raise FileNotFoundError("ewpd_out.yml not found. Check the previous cell output for the produced filename.")

with open(yml_path, "r") as f:
    Y = yaml.safe_load(f)

print("Top-level keys (first 80):")
keys = list(Y.keys())
print(keys[:80])
print("\nKey types/sizes (first 40):")
for k in keys[:40]:
    v = Y[k]
    if isinstance(v, dict):
        print(f"{k}: dict (len={len(v)})")
    elif isinstance(v, list):
        print(f"{k}: list (len={len(v)})")
    else:
        print(f"{k}: {type(v).__name__}")

## 5) Extract `coeff_names`, `A`, and `V`/`Vinv` (robust)

This cell searches the YAML for likely field names. If it fails, edit the manual override block.

In [ ]:
import numpy as np

def find_first_key(d, candidates):
    for c in candidates:
        if c in d:
            return c
    return None

COEF_CAND = ["coefficients","coefs","wilson_coefficients","wilson","parameters","pois","POIs","poi_names","wc_names","wc"]
A_CAND    = ["A","jacobian","J","linear","lin","response","smEFT_A","dobs_dC","dObs_dC","A_matrix"]
V_CAND    = ["cov","covariance","V","Vexp","Vtot","cov_tot","covariance_matrix","V_matrix"]
VI_CAND   = ["cov_inv","Vinv","V_inv","precision","invcov","covariance_inverse","V_inverse"]

coef_key = find_first_key(Y, COEF_CAND)
A_key    = find_first_key(Y, A_CAND)
V_key    = find_first_key(Y, V_CAND)
Vi_key   = find_first_key(Y, VI_CAND)

print("Auto-detected keys:", {"coef_key": coef_key, "A_key": A_key, "V_key": V_key, "Vi_key": Vi_key})

# -----------------------------
# Manual overrides (edit if needed)
# -----------------------------
# coef_key = "coefficients"
# A_key    = "A"
# V_key    = "covariance"
# Vi_key   = None

if coef_key is None or A_key is None or (V_key is None and Vi_key is None):
    print("\nAUTO-DETECTION FAILED or INCOMPLETE.")
    print("Set coef_key/A_key/V_key (or Vi_key) manually in the override block above.")
    raise ValueError("Missing required YAML fields.")

coeffs_obj = Y[coef_key]
if isinstance(coeffs_obj, dict):
    coeff_names = list(coeffs_obj.keys())
elif isinstance(coeffs_obj, list):
    coeff_names = [str(x) for x in coeffs_obj]
else:
    raise TypeError(f"Unexpected type for coefficients container: {type(coeffs_obj)}")

A = np.array(Y[A_key], dtype=float)

if Vi_key is not None:
    Vinv = np.array(Y[Vi_key], dtype=float)
else:
    V = np.array(Y[V_key], dtype=float)
    Vinv = np.linalg.pinv(V)

print("len(coeff_names) =", len(coeff_names))
print("A shape =", A.shape)
print("Vinv shape =", Vinv.shape)

if A.shape[1] != len(coeff_names):
    print("\nWARNING: A.shape[1] != len(coeff_names). Check YAML ordering/keys.")

## 6) Build coefficient-space Fisher matrix `F` and pseudoinverse covariance

\[
F = A^{T} V^{-1} A,\quad \Sigma_C = F^{+}.
\]

In [ ]:
F = A.T @ Vinv @ A
SigmaC = np.linalg.pinv(F)

svals = np.linalg.svd(F, compute_uv=False)
tol = max(F.shape) * np.max(svals) * 1e-12
rankF = int(np.sum(svals > tol))

print("F shape:", F.shape)
print("rank(F):", rankF, "out of", F.shape[0])
print("Smallest singular values (last 10):", svals[-10:])

## 7) Locate / set best-fit vector `C_hat`

If not present in the YAML, defaults to the zero vector.

In [ ]:
CHAT_CAND = ["C_hat","best_fit","bestfit","coeff_bestfit","mu_hat","central","Cbest","C0","C_SM"]
Chat_key = find_first_key(Y, CHAT_CAND)

import numpy as np
if Chat_key is not None:
    C_hat = np.array(Y[Chat_key], dtype=float).reshape(-1, 1)
    print("Found C_hat under key:", Chat_key)
else:
    C_hat = np.zeros((len(coeff_names), 1), dtype=float)
    print("No C_hat found in YAML; using C_hat = 0 vector.")

if C_hat.shape[0] != len(coeff_names):
    print("\nWARNING: C_hat length mismatch.")
    print("C_hat shape:", C_hat.shape, "len(coeff_names):", len(coeff_names))

## 8) Coefficient ordering table (paste `v_ray` in this ordering)

In [ ]:
import pandas as pd
display(pd.DataFrame({"i": range(len(coeff_names)), "coef": coeff_names}).head(120))
print("Total coefficients:", len(coeff_names))

# Paste YOUR ray here (same ordering as coeff_names)
v_ray = None  # e.g. [1.0, 0.0, -0.5, ...] length must match len(coeff_names)

## 9) Compute χ²⊥ using Fisher quadratic form

\[
\chi^2_{\perp} = \hat C^{T} F \hat C - \frac{(v^{T} F \hat C)^2}{v^{T} F v}.
\]

Effective degrees of freedom (rank-deficient case): \(\nu_{\mathrm{eff}} = \mathrm{rank}(F)-1\).

In [ ]:
def chi2_perp_from_F(C_hat, F, v_ray):
    C = np.asarray(C_hat, dtype=float).reshape(-1, 1)
    v = np.asarray(v_ray, dtype=float).reshape(-1, 1)
    num = float(v.T @ F @ C)
    den = float(v.T @ F @ v)
    if den <= 0:
        raise ValueError("Non-positive v^T F v. v may lie in a null direction or F is ill-conditioned.")
    chi2 = float(C.T @ F @ C - (num*num)/den)
    return chi2, num, den

if v_ray is None:
    print("Set v_ray in the previous cell and re-run.")
else:
    chi2, num, den = chi2_perp_from_F(C_hat, F, v_ray)
    nu_eff = max(rankF - 1, 0)
    print("chi2_perp =", chi2)
    print("rank(F)   =", rankF)
    print("nu_eff    =", nu_eff)
    print("v^T F C   =", num)
    print("v^T F v   =", den)

## 10) Diagnostics: fitted ray amplitude and largest residual components

In [ ]:
if v_ray is None:
    print("Set v_ray first.")
else:
    v = np.asarray(v_ray, dtype=float).reshape(-1, 1)
    C = np.asarray(C_hat, dtype=float).reshape(-1, 1)
    lambda_hat = float((v.T @ F @ C) / (v.T @ F @ v))
    C_par = lambda_hat * v
    C_perp = C - C_par

    df = pd.DataFrame({
        "coef": coeff_names,
        "C_hat": C.flatten(),
        "C_parallel": C_par.flatten(),
        "C_perp": C_perp.flatten(),
        "abs_C_perp": np.abs(C_perp.flatten())
    }).sort_values("abs_C_perp", ascending=False)

    print("lambda_hat =", lambda_hat)
    display(df.head(25))

## 11) Save outputs for download

In [ ]:
from pathlib import Path
np.save("coeff_names.npy", np.array(coeff_names, dtype=object))
np.save("A.npy", A)
np.save("Vinv.npy", Vinv)
np.save("F.npy", F)
np.save("SigmaC_pinv.npy", SigmaC)
np.save("C_hat.npy", C_hat)

print("Saved .npy files in:", Path(".").resolve())
!ls -lh *.npy

## 12) Download files (Colab)

Uncomment to download the saved arrays.

In [ ]:
# from google.colab import files
# for fn in ["coeff_names.npy","A.npy","Vinv.npy","F.npy","SigmaC_pinv.npy","C_hat.npy"]:
#     files.download(fn)